In [1]:
#pip install sciunit quantities efel scipy json2html neurom

In [2]:
#% matplotlib notebook
%matplotlib inline
from json2html import *
import quantities
from quantities import mV, nA
import IPython
import json
import collections
import pkg_resources
import numpy

#import sciunit
#from hippounit import models
from hippounit import tests
from hippounit.utils_50Ra_major_nmda import ModelLoader_synapse_on_Passive_Spine, ModelLoader_synapse_on_Active_Spine
from hippounit.utils_50Ra_major_nmda import ModelLoader_Modellke, ModelLoader_synapse_on_Existing_Spine
#from hippounit import capabilities
#import matplotlib.pyplot as plt

from neuron import h




Invalid MIT-MAGIC-COOKIE-1 key

In [3]:
import hippounit
print(hippounit.__file__)

/home/tarluca/Modellke_story/hippounit_Spine_project_20221102/hippounit/hippounit/__init__.py


# Synapses arrive at dendritic shaft - Active F-factor

## Instantiating the model

In [4]:
#all the outputs will be saved here. It will be an argument to the test.
"""Edit the line below"""
base_directory = '/mnt/csoport31-2/Modellezo_csapat/CA1_pyramidal/Spine_project_validation_results60/Basic_set_newopt/Shaft_active_F_factor_with_R_test/'

# path to mod files
"""Edit the line below"""
mod_files_path = "/home/tarluca/Modellke_story/Luca_modell_rework_2/prep_and_run_model_rework_new_K_stuff/nsg_csomag_model_rework_new/mod_files/"

# user function path
"""Edit the line below"""
user_function_path = '/home/tarluca/Modellke_story/Luca_modell_rework_2/prep_and_run_model_rework_new_K_stuff/nsg_csomag_model_rework_new/usr_fun_model_full_rework7.txt'

#Load cell model
model = ModelLoader_Modellke(mod_files_path = mod_files_path, user_function_path = user_function_path)

# path to hoc file
# the model must not display any GUI!!
"""Edit the line below"""
model.hocpath = "/home/tarluca/Modellke_story/Luca_modell_rework_2/prep_and_run_model_rework_new_K_stuff/nsg_csomag_model_rework_new/load_model_na_inhomo_minimal_model_full_soma_f_fact_true_diam_active_spine_KA_fact_50Ra.hoc"
# If the hoc file doesn't contain a template, this must be None (the default value is None)
model.template_name = None

# model.SomaSecList_name should be None, if there is no Section List in the model for the soma, or if the name of the soma section is given by setting model.soma (the default value is None)
model.SomaSecList_name = None
# if the soma is not in a section list or to use a specific somatic section, add its name here:
model.soma = 'soma'

# For the PSP Attenuation Test, and Back-propagating AP Test a section list containing the trunk sections is needed
model.TrunkSecList_name = 'trunk'
model.TuftSecList_name = 'tuft'
# For the Oblique Integration Test a section list containing the oblique dendritic sections is needed
model.ObliqueSecList_name = 'oblique_dendrites'

model.BasalSecList_name = 'basal_dendrites'

# It is important to set the v_init and the celsius parameters of the simulations here,
# as if they are only set in the model's files, they will be overwritten with the default values of the ModelLoader class.
# default values: v_init = -70, celsius = 34 
model.v_init = -70
model.celsius = 33

model.c_step_start = 0.000004
model.c_step_stop = 0.0000004
model.c_minmax = numpy.array([0.000004, 0.004])

## Backpropagating Action Potential Test

In [5]:
# Load target data
"""Edit the line below"""
with open('/home/tarluca/Modellke_story/hippounit_Spine_project_20221102/hippounit/target_features/feat_backpropagating_AP_target_data_ratio.json') as f:
    observation = json.load(f, object_pairs_hook=collections.OrderedDict)

IPython.display.HTML(json2html.convert(json = observation))

mean_AP1_amp_at_50um,0.8226
std_AP1_amp_at_50um,0.0573
mean_AP1_amp_at_150um,0.7645
std_AP1_amp_at_150um,0.1254
mean_AP1_amp_at_250um,0.7136
std_AP1_amp_at_250um,0.0896
mean_AP1_amp_strong_propagating_at_350um,0.6740
std_AP1_amp_strong_propagating_at_350um,0.0752
mean_AP1_amp_weak_propagating_at_350um,0.2015
std_AP1_amp_weak_propagating_at_350um,0.0519
mean_APlast_amp_at_50um,0.7755


In [6]:
#stimuli
import pkg_resources
stim_file = pkg_resources.resource_filename("hippounit", "tests/stimuli/bAP_stim/stim_bAP_test.json")
with open(stim_file, 'r') as f:
    config = json.load(f, object_pairs_hook=collections.OrderedDict)

In [7]:
# Instantiate the test class
test = tests.BackpropagatingAPTest2(observation = observation, config=config, save_all = True, force_run=True, force_run_FindCurrentStim=True, show_plot = True, base_directory = base_directory)

# Number of parallel processes
test.npool = 20

"""Edit the line below"""
f = open('/home/tarluca/Modellke_story/Luca_modell_rework_2/prep_and_run_model_rework_new_K_stuff/nsg_csomag_model_rework_new/results_ca1_results/last.csv')
lines = []
for l in f.readlines():
    #print l.strip().split(',')
    lines.append(l.strip().split(','))
f.close()

for i in range(1, len(lines)):
    parameters = [float(j) for j in lines[i]]
    #lines[i].pop(0)
    parameters.pop(0)
    print(parameters)
    
    model.parameters = parameters
    # outputs will be saved in folders named like this:
    """Edit the line below"""
    model.name="ca1_pc_"+str(i)
    if numpy.isnan(parameters[0]):
        print('Parameters are nan')
        pass
    else:
        try:
            #Run the test 
            score = test.judge(model)
            #Summarize and print the score achieved by the model on the test using SciUnit's summarize function
            score.summarize()

        except Exception as e:
            print('Model: ' + model.name + ' could not be run')
            print(e)
            pass


[-45.62129913381898, 0.015409599702098875, 0.16934641974325604, 0.102497419723226, 0.01715443774845717, 3.186132641811301e-06, 0.5652767116654946, -52.23667802503073, 0.00011082608622602389, 2.5294454453721573, 0.0005504949038343946, 0.004872936344012498, 0.08360930584022765, 4.477777340775202, 0.00027763058004307847, 1.9062470225446426, 3.090885414637565e-05, 0.00010916090173468279, -25.8501085422728, 8.658052647964135, 9.94749900407636]
numprocs=1
Dendritic locations to be tested (with their actual distances): {('dendrite[4]', 0.5): 44.010248981764285, ('dendrite[13]', 0.5): 45.759404409481654, ('dendrite[31]', 0.5): 51.874440048986514, ('dendrite[55]', 0.5): 64.04833266564478, ('dendrite[85]', 0.5): 147.38275623716865, ('dendrite[87]', 0.5): 158.77871747375843, ('dendrite[91]', 0.5): 166.10161451767556, ('dendrite[109]', 0.5): 232.2961368839031, ('dendrite[113]', 0.5): 247.82975267606994, ('dendrite[135]', 0.5): 335.58030914634867, ('dendrite[139]', 0.5): 344.2805632417561}
Finding 

Process ForkPoolWorker-15:
Process ForkPoolWorker-16:
Process ForkPoolWorker-20:
Process ForkPoolWorker-19:
Process ForkPoolWorker-18:
Process ForkPoolWorker-14:
Process ForkPoolWorker-21:
Process ForkPoolWorker-17:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/usr/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.8/multiprocessing/pool.py", line 114, in worker
    task = get()
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.ru

KeyboardInterrupt: 

Process ForkPoolWorker-3:
Traceback (most recent call last):
  File "/usr/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.8/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
  File "/usr/lib/python3.8/multiprocessing/pool.py", line 48, in mapstar
    return list(map(*args))
  File "/home/tarluca/Modellke_story/hippounit_Spine_project_20221102/hippounit/hippounit/tests/test_BackpropagatingAPTest2.py", line 273, in run_cclamp_on_soma
    t, v = model.get_vm(amp, delay, dur, section_stim, loc_stim, section_rec, loc_rec)
  File "/home/tarluca/Modellke_story/hippounit_Spine_project_20221102/hippounit/hippounit/capabilities/cap_ReceivesCurrentStimuli_ProvidesResponse.py", line 40, in get_vm
    t, v = self.inject_current(amp, delay, dur, section_stim, loc_stim, section_rec, loc_re

KeyboardInterrupt
Process ForkPoolWorker-12:
Traceback (most recent call last):
  File "/usr/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.8/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
  File "/usr/lib/python3.8/multiprocessing/pool.py", line 48, in mapstar
    return list(map(*args))
  File "/home/tarluca/Modellke_story/hippounit_Spine_project_20221102/hippounit/hippounit/tests/test_BackpropagatingAPTest2.py", line 273, in run_cclamp_on_soma
    t, v = model.get_vm(amp, delay, dur, section_stim, loc_stim, section_rec, loc_rec)
  File "/home/tarluca/Modellke_story/hippounit_Spine_project_20221102/hippounit/hippounit/capabilities/cap_ReceivesCurrentStimuli_ProvidesResponse.py", line 40, in get_vm
    t, v = self.inject_current(amp, delay, dur, section_stim, loc_stim, 